# A01 · 广告产品、参与方与交易链路

先修：公用31–34；无需广告领域先修。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

广告先是一种商业活动。广告主追求品牌或转化价值，媒体提供库存并承担体验成本，用户需要相关且不过度打扰的内容；平台协调资格、排序、交易和反馈。合约是预先约定库存与交付义务，竞价是请求级竞争；搜索描述需求入口，原生描述呈现形式，程序化描述交易方式，这些维度可以交叉。不能把所有广告等同于CTR预测。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
products=[{'product':'保量展示','allocation':'contract','billing':'CPM','risk':'欠投'},{'product':'搜索竞价','allocation':'auction','billing':'CPC','risk':'无关点击'},{'product':'程序化原生','allocation':'RTB','billing':'CPM','risk':'体验与隐私'}]
import pandas as pd
display(pd.DataFrame(products))
trace=ac.rtb()
print('请求 → 交易 → 反馈', trace['stages'])

,product,allocation,billing,risk
0,保量展示,contract,CPM,欠投
1,搜索竞价,auction,CPC,无关点击
2,程序化原生,RTB,CPM,体验与隐私


请求 → 交易 → 反馈 ['SSP request', 'ADX eligibility', 'DSP response', 'auction', 'win/loss notice', 'feedback']


## 解释与局限

同一个RTB请求由媒体/SSP供给、ADX组织、DSP代表需求参与。它只是构造链路，未发送广告网络请求。产品的结算事件可能晚于展示；品牌价值也不能用一次点击完全识别。

## 查看源码，再做练习

把搜索竞价改成保量搜索展示：哪些资格和结算规则保留，哪些需要改？为三方各列一项可能冲突的指标。

In [3]:
import inspect
print(inspect.getsource(ac.rtb))

def rtb(bid=1.2, competitor=1., reserve=.8, mechanism='second', timeout=False):
    if mechanism not in ('first','second'): raise ValueError('Invalid mechanism')
    win=not timeout and bid>=max(competitor,reserve)
    return dict(win=win, payment=(bid if mechanism=='first' else max(reserve,competitor)) if win else 0.,
                observed_market_price=competitor if win else None,
                lost_price_lower_bound=bid if not win and not timeout and bid>=reserve else None,
                stages=['SSP request','ADX eligibility','DSP response','auction','win/loss notice','feedback'])



## 来源与下一步

A-CCR 3–5、14–32页；A-LIU公开目录产品部分。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。